# Simplification Randomness

This notebook loads threshold-count data for each floating-point setting and compares the `union`, `train`, and `val` splits in a dynamic subplot grid.

## Set Paths and Imports

In [33]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"
color_seq = px.colors.qualitative.Dark24

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "simplification_randomness":
    NOTEBOOK_DIR = Path("experiments/simplification_randomness")

DATA_ROOT = NOTEBOOK_DIR.parent.parent / "experiment_data" / "simplification_randomness"
OUTPUT_ROOT = NOTEBOOK_DIR
OUTPUT_PATH = OUTPUT_ROOT / "simplification_threshold_counts.html"
STATIC_OUTPUT_PATH = OUTPUT_ROOT / "simplification_threshold_counts.png"
SPLITS = ("union", "train", "val")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_ROOT}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Reading data from: {DATA_ROOT}")

Reading data from: /home/santripta/Documents/losslandtopo/experiment_data/simplification_randomness


## Enumerate Floating-Point Data Folders

The source directory may contain either one subfolder per setting or flat files named `mnist-{setting}-{split}.csv`; both forms are detected.

In [34]:
float_pattern = re.compile(r"^-?(?:\d+(?:\.\d*)?|\.\d+)$")

setting_labels = set()
for path in DATA_ROOT.iterdir():
    if path.is_dir() and float_pattern.fullmatch(path.name):
        setting_labels.add(path.name)
    elif path.is_file() and path.suffix.lower() == ".csv":
        match = re.match(r"^mnist-(.+)-(union|train|val)\.csv$", path.name)
        if match and float_pattern.fullmatch(match.group(1)):
            setting_labels.add(match.group(1))

settings = sorted(setting_labels, key=float)
if not settings:
    raise FileNotFoundError(f"No floating-point settings found in {DATA_ROOT}")
print(f"Found settings: {', '.join(settings)}")

Found settings: 0.0, 0.05, 0.1, 0.5, 1.0


## Load Union/Train/Val Threshold-Count Data

## Prepare Sorted Threshold and Count Series

In [35]:
def source_path(setting, split):
    setting_dir = DATA_ROOT / setting
    candidates = (setting_dir / f"{split}.csv", setting_dir / f"mnist-{split}.csv", DATA_ROOT / f"mnist-{setting}-{split}.csv")
    return next((path for path in candidates if path.exists()), None)

def normalize_frame(frame, source):
    columns = {str(column).strip().lower(): column for column in frame.columns}
    threshold_column = columns.get("simplification threshold")
    count_column = columns.get("count")
    if threshold_column is None or count_column is None:
        warnings.warn(f"Skipping {source}: expected threshold and count columns")
        return pd.DataFrame(columns=["threshold", "count"])
    normalized = frame[[threshold_column, count_column]].rename(columns={threshold_column: "threshold", count_column: "count"})
    normalized["threshold"] = pd.to_numeric(normalized["threshold"], errors="coerce")
    normalized["count"] = pd.to_numeric(normalized["count"], errors="coerce")
    normalized = normalized.dropna().sort_values("threshold").reset_index(drop=True)
    if normalized.empty:
        warnings.warn(f"Skipping {source}: no valid numeric rows")
    return normalized

series = {setting: {} for setting in settings}
for setting in settings:
    for split in SPLITS:
        path = source_path(setting, split)
        if path is None:
            warnings.warn(f"Missing {split} data for setting {setting}")
            series[setting][split] = pd.DataFrame(columns=["threshold", "count"])
            continue
        try:
            series[setting][split] = normalize_frame(pd.read_csv(path), path)
        except (OSError, pd.errors.ParserError) as error:
            warnings.warn(f"Could not read {path}: {error}")
            series[setting][split] = pd.DataFrame(columns=["threshold", "count"])

print("Loaded rows:")
for setting in settings:
    counts = ", ".join(f"{split}={len(series[setting][split])}" for split in SPLITS)
    print(f"  {setting}: {counts}")

Loaded rows:
  0.0: union=289, train=261, val=49
  0.05: union=379, train=340, val=88
  0.1: union=381, train=345, val=94
  0.5: union=1057, train=926, val=103
  1.0: union=4122, train=3440, val=332


## Create Dynamic Subplot Grid (Rows = Float Names, Columns = union/train/val)

## Render Threshold-vs-Count Charts per Subplot

## Tight Layout, Titles, and Figure Export

In [36]:
nrows = len(settings)
fig = make_subplots(
    rows=nrows,
    cols=len(SPLITS),
    shared_xaxes="rows",
    shared_yaxes="rows",
    vertical_spacing=0.06,
    column_titles=[split.capitalize() for split in SPLITS]
)
split_colors = {split: color_seq[index] for index, split in enumerate(SPLITS)}

for row_index, randomness in enumerate(settings, start=1):
    for column_index, split in enumerate(SPLITS, start=1):
        frame = series[randomness][split]
        if frame.empty:
            fig.add_annotation(
                text="No data",
                x=0.5,
                y=0.5,
                showarrow=False,
                row=row_index,
                col=column_index,
            )
            continue
        fig.add_trace(
            go.Scatter(
                x=frame["threshold"],
                y=frame["count"],
                mode="lines",
                line=dict(shape="hv", color=split_colors[split], width=2),
                name=split.capitalize(),
                showlegend=False,
                hovertemplate="Threshold: %{x}<br>Count: %{y}<extra></extra>",
            ),
            row=row_index,
            col=column_index,
        )

fig.update_layout(
    title=dict(text="Simplification vs Valleys", x=0.5, xanchor="center"),
    margin=dict(l=105, r=20, t=70, b=55),
    width=300 * len(SPLITS),
    height=180 * nrows,
    font=dict(size=14),
    showlegend=False,
)
fig.update_xaxes(automargin=True, showgrid=True)
fig.update_yaxes(type="log", title_text="", automargin=True, showgrid=True)
for row_index, randomness in enumerate(settings, start=1):
    fig.update_yaxes(title_text=f"r={randomness}", row=row_index, col=1)

# fig.write_html(OUTPUT_PATH, include_plotlyjs="cdn")
try:
    fig.write_image(STATIC_OUTPUT_PATH, scale=4)
except (ValueError, ImportError) as error:
    warnings.warn(f"Static PNG export skipped; install kaleido to enable it: {error}")
print(f"Saved interactive plot to: {OUTPUT_PATH}")
fig.show()

Saved interactive plot to: /home/santripta/Documents/losslandtopo/experiments/simplification_randomness/simplification_threshold_counts.html


## Train and Val Comparison by Randomness

This compact figure overlays the train and validation tracks for each randomness setting.

In [41]:
SINGLE_OUTPUT_PATH = OUTPUT_ROOT / "simplification_threshold_train_val.html"
SINGLE_STATIC_OUTPUT_PATH = OUTPUT_ROOT / "simplification_threshold_train_val.png"

single_fig = make_subplots(
    rows=nrows,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.06,
)
track_colors = {"train": "rgba(225, 95, 153, 0.62)", "val": "rgba(28, 167, 28, 0.62)"}

for row_index, randomness in enumerate(settings, start=1):
    for split in ("train", "val"):
        frame = series[randomness][split]
        if frame.empty:
            continue
        single_fig.add_trace(
            go.Scatter(
                x=frame["threshold"],
                y=frame["count"],
                mode="lines",
                line=dict(shape="hv", color=track_colors[split], width=3),
                name=split.capitalize(),
                legendgroup=split,
                showlegend=row_index == 1,
                hovertemplate=f"Randomness: {randomness}<br>Threshold: %{{x}}<br>Count: %{{y}}<extra>{split.capitalize()}</extra>",
            ),
            row=row_index,
            col=1,
        )

single_fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
    width=600,
    height=180 * nrows,
    font=dict(size=14),
    legend=dict(
        x=0.98,
        y=0.995,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255, 255, 255, 0.72)"
    ),
)
single_fig.update_xaxes(title_text="", automargin=True, showgrid=True)
for row_index, randomness in enumerate(settings, start=1):
    single_fig.update_yaxes(type="log", title_text=f"r={randomness}", row=row_index, col=1, automargin=True, showgrid=True)

# single_fig.write_html(SINGLE_OUTPUT_PATH, include_plotlyjs="cdn")
try:
    single_fig.write_image(SINGLE_STATIC_OUTPUT_PATH, scale=4)
except (ValueError, ImportError) as error:
    warnings.warn(f"Static PNG export skipped; install kaleido to enable it: {error}")
print(f"Saved single-column comparison to: {SINGLE_OUTPUT_PATH}")
single_fig.show()

Saved single-column comparison to: /home/santripta/Documents/losslandtopo/experiments/simplification_randomness/simplification_threshold_train_val.html
